# 05 - Silver Pipeline Parcial

Este notebook documenta la capa **Silver** corregida: no crea facts ni dimensiones. Silver limpia, tipa, normaliza, deduplica, valida y publica datasets curados en Parquet para que Gold construya el modelo dimensional.


```mermaid
graph LR
  B["Bronze Parquet"] --> S["Silver Curated"]
  S --> M["municipalidades_curated"]
  S --> RE["renamu_curated"]
  S --> I["ingresos_municipales_curated"]
  S --> P["predial_esat_curated"]
  S --> R["sismepre_respuestas_curated"]
  S --> C["categorias_municipalidades_curated"]
  S --> Q["_quarantine"]
```


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('notebook-silver-evidence').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
silver = Path('/home/jovyan/work/data/silver')
audit = Path('/home/jovyan/work/data/audit')
expected = [
    'municipalidades_curated',
    'renamu_curated',
    'ingresos_municipales_curated',
    'predial_esat_curated',
    'sismepre_respuestas_curated',
    'sismepre_entidad_estado_curated',
    'sismepre_preguntas_curated',
    'sismepre_formularios_curated',
    'categorias_municipalidades_curated',
]
summary = []
for table in expected:
    path = silver / table
    df = spark.read.parquet(str(path))
    qpath = silver / '_quarantine' / table
    qrows = spark.read.parquet(str(qpath)).count() if qpath.exists() and list(qpath.rglob('*.parquet')) else 0
    summary.append((table, df.count(), len(df.columns), qrows))
spark.createDataFrame(summary, ['dataset_silver', 'registros', 'columnas', 'cuarentena']).show(50, truncate=False)


## Contrato Silver

- `ingresos_municipales_curated`: filtra `NIVEL_GOBIERNO = M`, tipa fechas/montos y conserva trazabilidad Bronze.
- `predial_esat_curated`: tipa métricas prediales y conserva granularidad correcta.
- `sismepre_respuestas_curated`: normaliza respuestas multivalor a formato largo.
- `municipalidades_curated`: consolida códigos y nombres municipales como dataset limpio, sin modelado dimensional.
- `renamu_curated`: tipa RENAMU desde Bronze Parquet, normaliza UBIGEO/geografía y publica personal/software tributario limpio para Gold.
- `categorias_municipalidades_curated`: normaliza categoría A-G del archivo del profesor y manda conflictos a cuarentena.


In [ ]:
quality = spark.read.option('recursiveFileLookup', 'true').option('multiLine', 'true').json(str(audit / 'quality_checks'))
quality.filter(F.col('check_name').startswith('silver_'))     .groupBy('dataset', 'check_type', 'status')     .count()     .orderBy('dataset', 'check_type', 'status')     .show(200, truncate=False)


## Migración Conceptual A Gold

Cualquier tabla con prefijo `fact_` o `dim_` pertenece a Gold. En esta versión Silver publica únicamente datasets curados; Gold toma esos contratos y construye `dim_*`, `fact_*` y `mart_*`.
